# 02 -- Extraction Research

Scratch space for validating Task 3's PDF-text-extraction pipeline against
real, downloaded PDFs before trusting it over the full corpus.

Like `01_crawler_research.ipynb`, this notebook imports from `src/extraction/`
rather than reimplementing extraction logic here.

## Project setup and imports

In [ ]:
import sys
from pathlib import Path

# Notebooks live in notebooks/, but the reusable pipeline code lives in
# src/ at the project root. Jupyter sets the working directory to wherever
# it was launched from, which isn't reliable -- so we search upward for
# requirements.txt (a stable marker of the project root) instead of
# hardcoding "..".
def find_project_root(marker="requirements.txt"):
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError("Could not locate project root (no requirements.txt found above cwd)")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


In [ ]:
from src.extraction import extract_text

extract_text


## Loading a sample PDF

Picks the first PDF found under `data/raw/` (any agency) so this notebook
works even before a specific agency has been fully crawled.

In [ ]:
sample_pdfs = sorted(extract_text.RAW_DIR.glob("*/*.pdf"))
print(f"{len(sample_pdfs)} PDFs available under data/raw/")

sample_pdf = sample_pdfs[0] if sample_pdfs else None
sample_pdf


## Calling the extraction functions from `src/extraction/`

In [ ]:
assert sample_pdf is not None, "Run the crawler (see 01_crawler_research.ipynb) before this notebook"

text = extract_text.extract_pdf_text(sample_pdf)
len(text)


## Displaying extracted text

In [ ]:
print(text[:1000])


## Checking page counts and text length

In [ ]:
import fitz

with fitz.open(sample_pdf) as doc:
    page_count = doc.page_count

word_count = len(text.split())
print(f"Pages: {page_count}")
print(f"Words: {word_count}")
print(f"Words per page: {word_count / page_count:.1f}")


## Detecting empty or scanned PDFs

Mirrors the `MIN_CHARS_FOR_SUCCESS` check in `extract_text.py` -- a PDF
that is scanned (image-only) will extract to little or no text and needs
OCR, which is explicitly out of scope for Task 3.

In [ ]:
is_likely_scanned = len(text.strip()) < extract_text.MIN_CHARS_FOR_SUCCESS
print(f"Likely scanned / needs OCR: {is_likely_scanned}")


## Comparing extraction quality across a few sample documents

In [ ]:
import pandas as pd

rows = []
for pdf_path in sample_pdfs[:10]:
    t = extract_text.extract_pdf_text(pdf_path)
    rows.append({
        "file": pdf_path.name,
        "chars": len(t),
        "words": len(t.split()),
        "likely_scanned": len(t.strip()) < extract_text.MIN_CHARS_FOR_SUCCESS,
    })

pd.DataFrame(rows)


## Recording extraction failures

In [ ]:
failures = extract_text.run()
failures
